# P8s 통일 프롬프트 & 채점 (v1)

팀 공통 규격. 아래 3개 상수는 **수정 금지**(수정 시 팀 전체 재실행 필요).

- `SYSTEM_PROMPT`
- `FEWSHOT` (3-shot, 전부 non-conflict = gold와 context 주장이 일치)
- `MAX_NEW_TOKENS = 48`

출력: `p8s_gen_{TAG}.csv` + K±xC± 2x2 집계.


## 0. 설정

In [ ]:
!pip -q install transformers accelerate

import os, json, re, string, unicodedata
import pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID   = "meta-llama/Llama-3.1-8B-Instruct"   # 전 실험 통일
DATA_PATH  = "p8s_pairs.csv"
TAG        = "base"        # 실험 구분자(예: base / steer_L13w_L17 / ffn_...)
MAX_NEW_TOKENS = 48        # 결정사항: 48토큰
BATCH_SIZE = 8
SEED = 0
torch.manual_seed(SEED)


## 1. 프롬프트 규격 (수정 금지)

### 왜 "single word"를 문자 그대로 쓰지 않는가
p8s의 gold 중 **67%가 2단어 이상**(`Milton Friedman`, `Tel Aviv`, `United States of America`).
`Answer with a single word`로 지시하면 모델이 성(姓)만 뱉거나 지시를 무시해 형식이 오히려 흔들린다.
따라서 **"정답 문자열만 출력, 문장/설명 금지"** 로 규정하고, few-shot으로 길이 감각을 잡는다.
채점상 효과는 single-word 지시와 동일하다(짧은 span만 나오므로).


In [ ]:
SYSTEM_PROMPT = (
    "You are a question answering system. "
    "Answer the question using ONLY the given context. "
    "Output the answer string itself and nothing else: no sentence, no explanation, "
    "no punctuation, no quotes. Keep it as short as possible (a name or a term). "
    "If the context does not state the answer, output the most likely short answer anyway."
)

# 3-shot. 전부 non-conflict(context가 주장하는 답 == gold).
# p8s / p8 어디에도 포함되지 않은 수기 예시 -> 데이터 누수 없음.
FEWSHOT = [
    {
        "context": ("Reykjavik Reykjavik is the capital and largest city of Iceland. "
                    "Located in the south-western part of the island, it is the seat of the "
                    "national government and the country's main cultural centre."),
        "question": "What is the capital of Iceland?",
        "answer": "Reykjavik",
    },
    {
        "context": ("The Old Man and the Sea The Old Man and the Sea is a short novel written by "
                    "Ernest Hemingway in 1951 in Cuba and published in 1952. It was the last major "
                    "work of fiction produced by Hemingway during his lifetime."),
        "question": "Who is the author of The Old Man and the Sea?",
        "answer": "Ernest Hemingway",
    },
    {
        "context": ("Ada Lovelace Augusta Ada King, Countess of Lovelace, was an English mathematician "
                    "chiefly known for her work on Charles Babbage's Analytical Engine. She is often "
                    "regarded as the first computer programmer."),
        "question": "What is Ada Lovelace's occupation?",
        "answer": "mathematician",
    },
]

USER_TMPL = "Context: {ctx}\nQuestion: {q}\nAnswer:"

def build_messages(question, context):
    """Llama-3.1-Instruct chat template용 messages. few-shot은 user/assistant 턴으로 넣는다."""
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in FEWSHOT:
        msgs.append({"role": "user",
                     "content": USER_TMPL.format(ctx=ex["context"], q=ex["question"])})
        msgs.append({"role": "assistant", "content": ex["answer"]})
    msgs.append({"role": "user", "content": USER_TMPL.format(ctx=context, q=question)})
    return msgs

# closed-book(K± 재판정용): context 없음, few-shot도 context 제거
SYSTEM_PROMPT_CB = (
    "You are a question answering system. Answer from your own knowledge. "
    "Output the answer string itself and nothing else: no sentence, no explanation, "
    "no punctuation, no quotes. Keep it as short as possible (a name or a term)."
)
USER_TMPL_CB = "Question: {q}\nAnswer:"

def build_messages_cb(question):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT_CB}]
    for ex in FEWSHOT:
        msgs.append({"role": "user", "content": USER_TMPL_CB.format(q=ex["question"])})
        msgs.append({"role": "assistant", "content": ex["answer"]})
    msgs.append({"role": "user", "content": USER_TMPL_CB.format(q=question)})
    return msgs

print(len(FEWSHOT), "shots")


## 2. 모델 로드

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
).eval()


## 3. 생성 (greedy, 48토큰) + 출력 절단

48토큰은 **여유분**이다. 형식이 깨져 문장이 나와도 잘라서 채점하도록 `truncate_answer()`로 1차 정리한다.


In [ ]:
@torch.no_grad()
def generate(msgs_list, max_new_tokens=MAX_NEW_TOKENS, batch_size=BATCH_SIZE):
    outs = []
    for i in range(0, len(msgs_list), batch_size):
        chunk = msgs_list[i:i+batch_size]
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                   for m in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens,
                             do_sample=False, temperature=None, top_p=None,
                             pad_token_id=tok.pad_token_id)
        for j in range(len(chunk)):
            new = gen[j][enc["input_ids"].shape[1]:]
            outs.append(tok.decode(new, skip_special_tokens=True))
        print(f"\r{min(i+batch_size, len(msgs_list))}/{len(msgs_list)}", end="")
    return outs

def truncate_answer(raw):
    """모델 출력 -> 채점용 짧은 span. 원문(raw)은 항상 따로 저장할 것."""
    s = raw.strip()
    s = s.split("\n")[0].strip()                  # 첫 줄만
    s = re.sub(r"^(answer|the answer is)\s*:?\s*", "", s, flags=re.I)
    s = s.split(". ")[0]                          # 문장화된 경우 첫 문장
    s = s.strip().strip('"\'')
    return re.sub(r"[.,;:\s]+$", "", s).strip()


## 4. 채점

- **주지표 `em`**: 정규화 후 gold 별칭 집합과 **완전일치**. 형식 지시를 건 실험이므로 이게 기본이다.
- **보조지표 `sub`**: gold 별칭이 출력에 **부분포함**(기존 PopQA/과거 실험 호환용).
- **`follows_ctx`**: 출력이 `answer_surface`(문맥 주장)와 일치하는지 → accept/retain 판정용.

C− 행에서 `em=1`이면 retain(파라메트릭 유지), `follows_ctx=1`이면 accept(문맥 추종)다.


In [ ]:
ARTICLES = {"a", "an", "the"}

def norm(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    toks = [t for t in s.split() if t not in ARTICLES]
    return " ".join(toks)

def score_row(pred, gold_list, answer_surface):
    p = norm(pred)
    golds = [norm(g) for g in gold_list]
    em  = int(any(p == g for g in golds))
    sub = int(any(g and g in p for g in golds))
    fc  = int(norm(answer_surface) in p) if isinstance(answer_surface, str) else 0
    return em, sub, fc


## 5. 실행

In [ ]:
df = pd.read_csv(DATA_PATH)
df["gold_list"] = df["gold"].apply(json.loads)

msgs_list = [build_messages(q, c) for q, c in zip(df["question"], df["ctx_text"])]
raw = generate(msgs_list)

df["raw_output"] = raw
df["pred"] = df["raw_output"].apply(truncate_answer)
df[["em", "sub", "follows_ctx"]] = pd.DataFrame(
    [score_row(p, g, a) for p, g, a in zip(df["pred"], df["gold_list"], df["answer_surface"])],
    index=df.index)

out = f"p8s_gen_{TAG}.csv"
df.drop(columns=["gold_list"]).to_csv(out, index=False)
print("\nsaved:", out)

# 형식 준수 체크(프롬프트가 먹었는지)
n_tok = df["pred"].str.split().str.len()
print("pred 평균 단어수:", round(n_tok.mean(), 2), "| 6단어 초과 비율:", round((n_tok > 6).mean(), 3))
print("EM:", round(df["em"].mean(), 3), "| SUB:", round(df["sub"].mean(), 3))


## 6. 2x2 집계 (결정사항 6번)

In [ ]:
piv = df.pivot_table(index="kside", columns="ctx_label",
                     values=["em", "follows_ctx"], aggfunc="mean").round(3)
print(piv)

# C- 조건에서의 accept/retain 분해
cneg = df[df["ctx_label"] == "C-"]
print("\n[C- only] accept(문맥추종) / retain(정답유지) rate")
print(cneg.groupby("kside")[["follows_ctx", "em"]].mean().round(3))


## 7. (필수 점검) K± 라벨 재판정

K± 라벨은 **이전 프롬프트 기준 closed-book 판정** 결과다. 답변형식 프롬프트를 바꾸면 closed-book 정확도도 바뀌므로,
기존 K± 라벨이 그대로 유효하다는 보장이 없다. 아래로 flip rate를 먼저 확인하고, 큰 폭이면 라벨을 갱신한 뒤 본 실험을 해석한다.
(질문 612개뿐이라 저비용이다.)


In [ ]:
q = df.drop_duplicates("question")[["question", "gold_list", "kside"]].reset_index(drop=True)
cb_raw = generate([build_messages_cb(x) for x in q["question"]])
q["cb_pred"] = [truncate_answer(r) for r in cb_raw]
q["cb_em"] = [score_row(p, g, "")[0] for p, g in zip(q["cb_pred"], q["gold_list"])]
q["kside_new"] = q["cb_em"].map({1: "K+", 0: "K−"})

flip = (q["kside"] != q["kside_new"]).mean()
print("flip rate:", round(flip, 3))
print(pd.crosstab(q["kside"], q["kside_new"]))
q.drop(columns=["gold_list"]).to_csv(f"p8s_klabel_recheck_{TAG}.csv", index=False)
